In [1]:
!pip install openai python-dotenv langgraph

  Obtaining dependency information for langgraph from https://files.pythonhosted.org/packages/48/9e/31ca236104966d7bb14ea9e93cfd73350aea8c41008ddf057b65794ed10d/langgraph-1.2.4-py3-none-any.whl.metadata
  Obtaining dependency information for langchain-core<2,>=1.4.0 from https://files.pythonhosted.org/packages/ca/79/531d8ee5dc5bf464c18cc86b087569307bc2d6b74548753f26122d08746d/langchain_core-1.4.1-py3-none-any.whl.metadata
  Obtaining dependency information for langgraph-checkpoint<5.0.0,>=4.1.0 from https://files.pythonhosted.org/packages/bd/b4/71425e3e38be92611300b9cc5e46a5bf98ab23f5ea8a75b73d02a2f1413c/langgraph_checkpoint-4.1.1-py3-none-any.whl.metadata
  Obtaining dependency information for langgraph-prebuilt<1.2.0,>=1.1.0 from https://files.pythonhosted.org/packages/e9/43/3fe1a700b8490ed02679cdbbc8c915eb23a092faf496c9c1118abcd10be3/langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata
  Obtaining dependency information for langgraph-sdk<0.5.0,>=0.4.2 from https://files.pythonhosted.o

In [3]:
import os
import json
import base64
from pathlib import Path
from typing import Any, TypedDict

from dotenv import load_dotenv
from openai import OpenAI

from langgraph.graph import START, END, StateGraph

In [5]:
load_dotenv(override=True)
DASHSCOPE_API_KEY = os.getenv("DASHSCOPE_API_KEY")

# print(DASHSCOPE_API_KEY)

In [6]:
ALIBABA_BASE_URL = "https://dashscope-intl.aliyuncs.com/compatible-mode/v1"

QWEN_MODEL = "qwen3.5-plus"

client = OpenAI(
    api_key=DASHSCOPE_API_KEY,
    base_url=ALIBABA_BASE_URL
)

In [ ]:
# переводим картинку в последовательность байтов

def image_file_processing(path: str) -> str:
    path = Path(path)
    mime_type = 'image/png'
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')

    return f"data:{mime_type};base64,{encoded}"

# Список доступных тулзов

In [8]:
AVAILABLE_TOOLS = [
    {
        "name": "adjust_brightness",
        "description": "Изменяет яркость изображения.",
        "args": {"factor": "float, например 1.1"},
    },
    {
        "name": "adjust_contrast",
        "description": "Изменяет контраст изображения.",
        "args": {"factor": "float, например 1.2"},
    },
    {
        "name": "adjust_saturation",
        "description": "Изменяет насыщенность изображения.",
        "args": {"factor": "float, например 1.15"},
    },
    {
        "name": "sharpen",
        "description": "Повышает резкость изображения.",
        "args": {"factor": "float, например 1.3"},
    },
    {
        "name": "gaussian_blur",
        "description": "Применяет гауссово размытие.",
        "args": {"kernel_size": "int odd, например 5"},
    },
    {
        "name": "median_blur",
        "description": "Применяет медианный фильтр.",
        "args": {"kernel_size": "int odd, например 5"},
    },
    {
        "name": "to_grayscale",
        "description": "Переводит изображение в градации серого.",
        "args": {},
    },
    {
        "name": "threshold_binary",
        "description": "Бинаризует изображение по порогу.",
        "args": {"threshold": "int 0..255"},
    },
    {
        "name": "canny_edges",
        "description": "Выделяет границы объектов методом Canny.",
        "args": {"threshold1": "int", "threshold2": "int"},
    },
    {
        "name": "add_text",
        "description": "Добавляет текст на изображение.",
        "args": {
            "text": "str",
            "position": "top | center | bottom",
            "color": "str",
        },
    },
    {
        "name": "segment_object",
        "description": "Находит маску объекта по текстовому описанию.",
        "args": {"target": "str, например 'person on the right'"},
    },
    {
        "name": "lama_inpaint",
        "description": "Удаляет объект по маске через LaMa inpainting.",
        "args": {"mask_source": "previous_step"},
    },
    {
        "name": "darken_background",
        "description": "Затемняет фон вокруг объекта по маске.",
        "args": {"factor": "float, например 0.5"},
    },
    {
        "name": "match_color_statistics",
        "description": "Делает второе изображение похожим на первое по цветовой статистике.",
        "args": {},
    },
]

# Промпты

In [ ]:
# PLANNER_AGENT_PROMPT = f"""
# Ты — planner для агента редактирования изображений.

# Тебе дан пользовательский запрос и одно или два изображения.
# Твоя задача — НЕ редактировать изображение самостоятельно, а выбрать последовательность инструментов.
# Доступные инструменты:
# {json.dumps(available_tools, ensure_ascii=False, indent=2)}
# Верни строго JSON без markdown и без пояснений вне JSON.

# Формат:
# {{
#   "task_type": "one of: enhancement, add_text, object_removal, local_edit, two_image_style_transfer, unknown",
#   "needs_second_image": true,
#   "actions": [
#     {{
#       "tool": "tool_name",
#       "args": {{}},
#       "reason": "почему выбран этот инструмент"
#     }}
#   ],
#   "user_message": "короткое сообщение пользователю о плане"
# }}

# Правила:
# - Используй только tools из списка доступных инструментов.
# - Если нужно удалить объект, выбери segment_object и lama_inpaint.
# - Если нужно добавить текст, выбери add_text.
# - Если нужно сделать фото как в журнале, выбери adjust_brightness, adjust_contrast, adjust_saturation, sharpen.
# - Если нужно сделать второе изображение похожим на первое, выбери match_color_statistics.
# - Если нужно выделить объект и затемнить фон, выбери segment_object и darken_background.
# - Если запрос неясный, task_type="unknown" и actions=[].
# - Для удаления объекта в args у segment_object положи target на английском, например "person on the right".
# - Не используй tools, которых нет в списке.
# Пользовательский запрос:
# {user_instruction}
# """

# вызов модели(у нас квен какой-то, можно будет менять)

In [12]:
def call_qwen_planner(
        user_instruction: str,
        image_paths: list[str],
        available_tools: list[dict],
        model: str = QWEN_MODEL,
) -> str:
    content = []
    for i in image_paths:
        temp_pic = image_file_processing(i)
        content.append(
            {
                'type': 'image_url',
                'image_url': {'url': temp_pic},
            }
        )

    planner_agent_prompt = f"""
        Ты — planner для агента редактирования изображений.

        Тебе дан пользовательский запрос и одно или два изображения.
        Твоя задача — НЕ редактировать изображение самостоятельно, а выбрать последовательность инструментов.
        Доступные инструменты:
        {json.dumps(available_tools, ensure_ascii=False, indent=2)}
        Верни строго JSON без markdown и без пояснений вне JSON.

        Формат:
        {{
        "task_type": "one of: enhancement, add_text, object_removal, local_edit, two_image_style_transfer, unknown",
        "needs_second_image": true,
        "actions": [
            {{
            "tool": "tool_name",
            "args": {{}},
            "reason": "почему выбран этот инструмент"
            }}
        ],
        "user_message": "короткое сообщение пользователю о плане"
        }}

        Правила:
        - Используй только tools из списка доступных инструментов.
        - Если нужно удалить объект, выбери segment_object и lama_inpaint.
        - Если нужно добавить текст, выбери add_text.
        - Если нужно сделать фото как в журнале, выбери adjust_brightness, adjust_contrast, adjust_saturation, sharpen.
        - Если нужно сделать второе изображение похожим на первое, выбери match_color_statistics.
        - Если нужно выделить объект и затемнить фон, выбери segment_object и darken_background.
        - Если запрос неясный, task_type="unknown" и actions=[].
        - Для удаления объекта в args у segment_object положи target на английском, например "person on the right".
        - Не используй tools, которых нет в списке.
        Пользовательский запрос:
        {user_instruction}
    """

    content.append(
        {
            'type': 'text',
            'text': planner_agent_prompt,
        }
    )

    response = client.chat.completions.create(
        model=model,
        messages=[
            {
                'role': 'system',
                'content': 'Ты - аккуратный VLM планировщик для редактирования изображений, отвечай только валидным JSON.'
            },

            {
                'role': 'user',
                'content': content,
            }
        ],
        temperature=0.0,
        max_tokens=1200,
    )

    return response.choices[0].message.content

In [13]:
raw = call_qwen_planner(
    'убери сахарную вату',
    ['assets/tanin_jazz.png'],
    AVAILABLE_TOOLS,
)

In [14]:
print(raw)

{
  "task_type": "object_removal",
  "needs_second_image": false,
  "actions": [
    {
      "tool": "segment_object",
      "args": {
        "target": "pink cotton candy"
      },
      "reason": "Создать маску сахарной ваты для последующего удаления"
    },
    {
      "tool": "lama_inpaint",
      "args": {
        "mask_source": "previous_step"
      },
      "reason": "Удалить сахарную вату используя маску из предыдущего шага"
    }
  ],
  "user_message": "Удалю сахарную вату с изображения: сначала выделю объект, затем заполню область фоном"
}


# State for Langgraph

In [15]:
class PlannerState(TypedDict, total=False):
    user_instruction: str
    image_paths: list[str]
    available_tools: list[dict]

    raw_planner_response: str
    plan: dict[str, Any]

    logs: list[dict[str, Any]]
    error: str | None

# Nodes

In [16]:
def prepare_input_node(state: PlannerState) -> PlannerState:
    logs = state.get("logs", [])

    image_paths = state.get("image_paths", [])

    if len(image_paths) == 0:
        raise ValueError("At least one image is required.")

    if len(image_paths) > 2:
        raise ValueError("Only up to 2 images are supported.")

    logs.append(
        {
            "node": "prepare_input",
            "status": "ok",
            "message": f"Received {len(image_paths)} image(s).",
            "image_paths": image_paths,
            "user_instruction": state["user_instruction"],
        }
    )

    return {
        **state,
        "available_tools": AVAILABLE_TOOLS,
        "logs": logs,
        "error": None,
    }

In [17]:
def qwen_planner_node(state: PlannerState) -> PlannerState:
    logs = state.get("logs", [])

    raw_response = call_qwen_planner(
        user_instruction=state["user_instruction"],
        image_paths=state["image_paths"],
        available_tools=state["available_tools"],
    )

    logs.append(
        {
            "node": "qwen_planner",
            "status": "ok",
            "model": QWEN_MODEL,
            "raw_response": raw_response,
        }
    )

    return {
        **state,
        "raw_planner_response": raw_response,
        "logs": logs,
    }

In [18]:
def parse_plan_node(state: PlannerState) -> PlannerState:
    logs = state.get("logs", [])
    raw = state["raw_planner_response"]

    try:
        plan = json.loads(raw)
        status = "ok"
        error = None
    except json.JSONDecodeError as exc:
        plan = {
            "task_type": "unknown",
            "needs_second_image": False,
            "actions": [],
            "user_message": "Не удалось распарсить JSON-план от модели.",
        }
        status = "failed"
        error = str(exc)

    logs.append(
        {
            "node": "parse_plan",
            "status": status,
            "plan": plan,
            "error": error,
        }
    )

    return {
        **state,
        "plan": plan,
        "logs": logs,
        "error": error,
    }

In [19]:
def finish_node(state: PlannerState) -> PlannerState:
    logs = state.get("logs", [])

    logs.append(
        {
            "node": "finish",
            "status": "ok",
            "message": "Planner graph finished.",
        }
    )

    return {
        **state,
        "logs": logs,
    }

In [20]:
graph = StateGraph(PlannerState)

graph.add_node("prepare_input", prepare_input_node)
graph.add_node("qwen_planner", qwen_planner_node)
graph.add_node("parse_plan", parse_plan_node)
graph.add_node("finish", finish_node)

graph.add_edge(START, "prepare_input")
graph.add_edge("prepare_input", "qwen_planner")
graph.add_edge("qwen_planner", "parse_plan")
graph.add_edge("parse_plan", "finish")
graph.add_edge("finish", END)

planner_app = graph.compile()

In [22]:
result = planner_app.invoke(
    {
        "user_instruction": "Убери человека справа с фотографии.",
        "image_paths": ["assets/tanin_jazz.png"],
        "logs": [],
    }
)

print("=== PLAN ===")
print(json.dumps(result["plan"], ensure_ascii=False, indent=2))

print("\n=== LOGS ===")
print(json.dumps(result["logs"], ensure_ascii=False, indent=2))

=== PLAN ===
{
  "task_type": "object_removal",
  "needs_second_image": false,
  "actions": [
    {
      "tool": "segment_object",
      "args": {
        "target": "person on the right"
      },
      "reason": "Выделение человека справа на изображении для создания маски удаления"
    },
    {
      "tool": "lama_inpaint",
      "args": {
        "mask_source": "previous_step"
      },
      "reason": "Удаление выделенного объекта и восстановление фона с помощью инпейнтинга"
    }
  ],
  "user_message": "Я удалю человека справа с фотографии, сначала выделив его, а затем заполнив фон."
}

=== LOGS ===
[
  {
    "node": "prepare_input",
    "status": "ok",
    "message": "Received 1 image(s).",
    "image_paths": [
      "assets/tanin_jazz.png"
    ],
    "user_instruction": "Убери человека справа с фотографии."
  },
  {
    "node": "qwen_planner",
    "status": "ok",
    "model": "qwen3.5-plus",
    "raw_response": "{\n  \"task_type\": \"object_removal\",\n  \"needs_second_image\": fa